# OTFLEX Gavin Workflow (Operation-Based Notebook)

This notebook executes the full experimental sequence directly in notebook cells, without graph traversal.

Execution model here:
- Each workflow action is a separate runnable code cell.
- Parameters are declared at the top of each action cell for easy edits.
- Cells are order-agnostic and can be rearranged.
- If something fails, jump to the teardown cell to leave devices in a safe state.

In [ ]:
import asyncio
import json
import subprocess
import sys
import time
from pathlib import Path

# Find repo root so imports from src.* work no matter where notebook launches from.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not find repository root containing src/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.adapters.iot_mqtt import (
    ControllerBeacon,
    FurnaceMQTT,
    PumpMQTT,
    ReactorMQTT,
    _best_effort_all_off,
)
from src.adapters.otflex_adapter import OTFlex
from src.core.opentrons import opentronsClient

print(f"Repo root: {repo_root}")

# Device configs defined directly in this notebook
otflex_cfg = {
    "module": "otflex_runtime.py",
    "controller_ip": "169.254.179.32",
    "deck": {
        "slots": {
            "A1": {"name": "single_electrode_module", "display": "3 electrodes", "labware": "sdl1_single_electrode_tiprack", "file": "data/labware_definitions/sdl1_single_electrode_tiprack.json", "slot_label": "A1"},
            "A2": {"name": "parallel_electrode_module", "display": "4 electrode tool", "labware": "sdl1_parallel_electrode_tiprack", "file": "data/labware_definitions/sdl1_parallel_electrode_tiprack.json", "slot_label": "A2"},
            "A3": {"name": "trash", "display": "trash", "labware": "opentrons_flex_trash", "slot_label": "A3"},
            "B1": {"name": "autreactor", "display": "autreactor", "labware": "actuated_reactor", "file": "data/labware_definitions/actuated_reactor.json", "slot_label": "B1"},
            "B2": {"name": "precursor_solutions", "display": "precursor solutions", "labware": "sdl1_11_vials_20mL", "file": "data/labware_definitions/sdl1_11_vials_20mL.json", "slot_label": "B2"},
            "C3": {"name": "tiprack_1000ul", "display": "1000ul tips", "labware": "opentrons_flex_96_tiprack_1000ul", "slot_label": "C3"},
            "D1": {"name": "substrate_tower", "display": "tower of substrates", "labware": "zou_21_wellplate_4500ul", "file": "data/labware_definitions/zou_21_wellplate_4500ul.json", "slot_label": "D1"}
        },
        "pipettes": {"right": {"model": "p1000_single_flex"}}
    }
}

mqtt_cfg = {
    "broker": "localhost",
    "port": 1883,
    "username": "pyctl-controller",
    "password": "controller",
    "topics": {
        "pumps": "pumps/01",
        "reactor": "reactor/01",
        "furnace": "furnace/01",
    },
}

# Critical: flushWell uses OTFlex runtime MQTT helpers, so pass MQTT config into otflex runtime.
otflex_cfg["mqtt"] = mqtt_cfg

opentrons_cfg = {
    "ip": otflex_cfg["controller_ip"],
    "robot": "flex",
}

# Shared runtime objects.
otflex = OTFlex(otflex_cfg, root_dir=repo_root)

beacon = None
pumps = None
reactor = None
furnace = None

print("Setup objects created. Run homing cell next, then connect devices.")

## Home Opentrons Robot First

Run this before any workflow action.

This cell inlines the same approach used by `scripts/home_opentrons.py` (directly using `opentronsClient`), without importing that script.

In [ ]:
# Homing params (editable)
homing_params = {
    "ip": opentrons_cfg["ip"],
    "robot": opentrons_cfg["robot"],
}

client = opentronsClient(
    strRobotIP=homing_params["ip"],
    strRobot=homing_params["robot"],
)
client.homeRobot()
print(f"Homed {homing_params['robot']} at {homing_params['ip']}")

## Capture Lab Setup Image (Before Connect)

Run this after homing and before connecting devices.

This uses the same SSH capture flow as the verbose Pi camera notebook:
- connect to Raspberry Pi over SSH
- capture an image with Picamera2
- download to `data/out/images`
- rotate the image 180 degrees
- display the image inline
- remove temporary remote image

In [ ]:
# Camera params (editable)
camera_params = {
    "host": "192.168.0.101",
    "username": "sdl1",
    "password": "1144",
    "ssh_port": 22,
    "connect_timeout_s": 8,
    "connect_retries": 3,
    "remote_capture_dir": "/tmp",
    "warmup_s": 2,
    "rotate_degrees": 180,
}

from datetime import datetime
import socket

try:
    import paramiko
except Exception as exc:
    raise ImportError(
        "paramiko is required for SSH camera capture. Install it in the notebook environment."
    ) from exc

try:
    from PIL import Image as PILImage
    from IPython.display import Image as IPyImage, display
except Exception as exc:
    raise ImportError(
        "Pillow and IPython display are required for rotate/display. Install pillow in the notebook environment."
    ) from exc

out_dir = repo_root / "data" / "out" / "images"
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"otflex-top_{timestamp}.jpg"
remote_path = f"{camera_params['remote_capture_dir']}/{filename}"
local_path = out_dir / filename

with socket.create_connection(
    (camera_params["host"], int(camera_params["ssh_port"])),
    timeout=3,
    ):
    pass
print(f"SSH port reachable at {camera_params['host']}:{camera_params['ssh_port']}")

ssh = None
try:
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    for attempt in range(1, int(camera_params["connect_retries"]) + 1):
        try:
            print(f"SSH connect attempt {attempt}/{camera_params['connect_retries']}")
            ssh.connect(
                camera_params["host"],
                port=int(camera_params["ssh_port"]),
                username=camera_params["username"],
                password=camera_params["password"],
                timeout=float(camera_params["connect_timeout_s"]),
                banner_timeout=float(camera_params["connect_timeout_s"]),
                auth_timeout=float(camera_params["connect_timeout_s"]),
            )
            break
        except (socket.timeout, TimeoutError):
            if attempt == int(camera_params["connect_retries"]):
                raise
            print("Timeout; retrying...")

    capture_cmd = f"""python3 << 'EOF'
from picamera2 import Picamera2
import time

picam2 = Picamera2()
config = picam2.create_still_configuration(main={{\"size\": (2028, 1520)}})
picam2.configure(config)
picam2.start()
time.sleep({float(camera_params['warmup_s'])})
picam2.capture_file(\"{remote_path}\")
picam2.close()
print(\"OK\")
EOF
"""

    _, stdout, stderr = ssh.exec_command(capture_cmd, timeout=45)
    exit_code = stdout.channel.recv_exit_status()
    err_text = stderr.read().decode()
    if exit_code != 0:
        raise RuntimeError(f"Pi capture failed: {err_text}")

    sftp = ssh.open_sftp()
    sftp.get(remote_path, str(local_path))
    sftp.close()
    ssh.exec_command(f"rm {remote_path}")

    print(f"Lab setup image saved: {local_path}")

    # Rotate image to match top-down orientation used in verbose notebook.
    with PILImage.open(local_path) as img:
        rotated = img.rotate(int(camera_params["rotate_degrees"]), expand=True)
        rotated.save(local_path)

    print(f"Showing rotated image: {local_path.name}")
    display(IPyImage(filename=str(local_path)))
except Exception:
    raise
finally:
    if ssh is not None:
        ssh.close()

## Connect Devices

This cell mirrors runner connect behavior:
- connect OTFlex
- start controller beacon
- connect MQTT devices used by this workflow (pump/reactor/furnace)

In [ ]:
# Connection params (editable)
connect_params = {
    "settle_s": 0.5,
}

await otflex.connect()

beacon = ControllerBeacon(
    broker=mqtt_cfg["broker"],
    port=mqtt_cfg["port"],
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
    client_id="nb-controller",
    status_topic="pyctl/status",
    heartbeat_topic="pyctl/heartbeat",
)
beacon.start()

common = dict(
    broker=mqtt_cfg["broker"],
    port=mqtt_cfg["port"],
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
)

pumps = PumpMQTT(**common, base_topic=mqtt_cfg["topics"]["pumps"], client_id="nb-pumps")
reactor = ReactorMQTT(**common, base_topic=mqtt_cfg["topics"]["reactor"], client_id="nb-reactor")
furnace = FurnaceMQTT(**common, base_topic=mqtt_cfg["topics"]["furnace"], client_id="nb-furnace")

pumps.ensure_connected()
reactor.ensure_connected()
furnace.ensure_connected()

# Ensure flushWell and other OTFlex MQTT helpers use known-good notebook clients.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is not None:
    rt.mqtt_pumps = pumps
    rt.mqtt_reactor = reactor
    rt.mqtt_furnace = furnace
    print("Bound OTFlex runtime MQTT handles to notebook MQTT clients.")

time.sleep(connect_params["settle_s"])
print("All configured devices connected.")

## Raise Autoreactor (mqttReactor)

Publishes `FORWARD:5000` to the reactor topic and waits for firmware auto-stop timing.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Run xArm Plate-to-Reactor Transfer

This action replaces the fixed pause.

The notebook waits until `src/core/plate2reactor.py` completes, then the next action can lower the reactor.

In [ ]:
# Action params (editable)
params = {
    "script_relpath": "src/core/plate2reactor.py",
    "python_exe": sys.executable,
    "timeout_s": 0,  # 0 means no timeout
}

script_path = (repo_root / params["script_relpath"]).resolve()
if not script_path.exists():
    raise FileNotFoundError(f"xArm script not found: {script_path}")

cmd = [params["python_exe"], str(script_path)]
print("Running xArm transfer script:", " ".join(cmd))

run_kwargs = dict(
    cwd=str(repo_root),
    text=True,
    capture_output=True,
    check=False,
    timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
)

try:
    result = subprocess.run(cmd, **run_kwargs)
except subprocess.TimeoutExpired as exc:
    raise TimeoutError(
        f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
    ) from exc

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"xArm transfer failed with exit code {result.returncode}: {script_path}"
    )

print("xArm transfer action complete.")

## Lower Autoreactor (mqttReactor)

Publishes `REVERSE:8000` and waits for duration alignment.

In [ ]:
# Action params (editable)
params = {
    "direction": "reverse",
    "duration_ms": 8000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor lower action complete.")

## Transfer Precursor A1 (otflexTransfer)

Transfers precursor from `precursor_solutions.A1` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "precursor_solutions", "well": "A1", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## Transfer Precursor A2 (otflexTransfer)

Transfers precursor from `precursor_solutions.A2` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "precursor_solutions", "well": "A2", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## Tool Transfer (otflexToolTransfer)

Picks the 4-electrode tool and places it into reactor column 1.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "parallel_electrode_module", "well": "A1", "offsetX": 0.5, "offsetY": 1.0, "offsetZ": 0.0},
    "to": {"labware": "autreactor", "well": "A1", "offsetX": -3.5, "offsetY": -34.5, "offsetZ": 10.0},
    "pipette": "p1000_single_flex",
    "move_speed": 80,
    "approach_offset_z": 0.0,
    "insert_pause_s": 1.0,
    "return_dZ": 12.0,
}

await otflex.toolTransfer(params)
print("Tool transfer action complete.")

## Flush Tool Wells (otflexFlushWell)

Flushes wells `A1` and `B1` using pump IDs configured in params.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "single_electrode_module", "well": "C1", "offsetX": 0.0, "offsetY": 0.0, "offsetZ": 0.0},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 5.0},
    "pipette": "p1000_single_flex",
    "in_pump_id": 1,
    "out_pump_id": 3,
    "time_ms": 3000,
    "repeats": 2,
    "purge_ms": 0,
    "return_dZ": 12.0,
}

# Preflight: flushWell relies on OTFlex runtime MQTT pump handle.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or rt.mqtt_pumps is None:
    raise RuntimeError(
        "OTFlex runtime MQTT pumps are not configured. Run the Connect Devices cell first."
    )

await otflex.flushWell(params)
print("Flush action complete.")

## Raise Autoreactor Final (mqttReactor)

Final reactor raise before furnace sequence.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Open Furnace Door (mqttFurnace)

Publishes `OPEN:6400` and waits to stay aligned with movement duration.

In [ ]:
# Action params (editable)
# params = {
#     "action": "open",
#     "duration_ms": 6400,
# }

# if params["action"].lower() == "open":
#     await asyncio.to_thread(furnace.open, params["duration_ms"])
# else:
#     await asyncio.to_thread(furnace.close, params["duration_ms"])
# await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
# print("Furnace open action complete.")

## Run xArm Reactor-to-Furnace Transfer

This action replaces the fixed post-open pause.

The notebook waits until `src/core/reactor2furnace.py` completes, then the next action can close the furnace door.

In [ ]:
# Action params (editable)
# params = {
#     "script_relpath": "src/core/reactor2furnace.py",
#     "python_exe": sys.executable,
#     "timeout_s": 0,  # 0 means no timeout
# }

# script_path = (repo_root / params["script_relpath"]).resolve()
# if not script_path.exists():
#     raise FileNotFoundError(f"xArm script not found: {script_path}")

# cmd = [params["python_exe"], str(script_path)]
# print("Running xArm transfer script:", " ".join(cmd))

# run_kwargs = dict(
#     cwd=str(repo_root),
#     text=True,
#     capture_output=True,
#     check=False,
#     timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
# )

# try:
#     result = subprocess.run(cmd, **run_kwargs)
# except subprocess.TimeoutExpired as exc:
#     raise TimeoutError(
#         f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
#     ) from exc

# if result.stdout:
#     print(result.stdout)
# if result.stderr:
#     print(result.stderr)

# if result.returncode != 0:
#     raise RuntimeError(
#         f"xArm transfer failed with exit code {result.returncode}: {script_path}"
#     )

# print("xArm reactor-to-furnace action complete.")

## Close Furnace Door (mqttFurnace)

Publishes `CLOSE:10000` and waits to align with door motion.

In [ ]:
# Action params (editable)
# params = {
#     "action": "close",
#     "duration_ms": 10000,
# }

# if params["action"].lower() == "open":
#     await asyncio.to_thread(furnace.open, params["duration_ms"])
# else:
#     await asyncio.to_thread(furnace.close, params["duration_ms"])
# await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
# print("Furnace close action complete.")

## End Marker

No hardware action. This marks the workflow end.

In [ ]:
print("Workflow sequence reached end marker.")

## Teardown - Safe Shutdown

Run this at the end, or immediately if any step errors.

This mirrors the runner's best-effort shutdown intent:
- turn off active MQTT outputs
- disconnect MQTT clients and beacon
- disconnect OTFlex

In [ ]:
try:
    _best_effort_all_off(pumps=pumps, reactor=reactor, furnace=furnace)
except Exception as exc:
    print(f"Best-effort OFF warning: {exc}")

for dev, name in ((pumps, "pumps"), (reactor, "reactor"), (furnace, "furnace")):
    if dev is not None:
        try:
            dev.disconnect()
        except Exception as exc:
            print(f"Disconnect warning ({name}): {exc}")

if beacon is not None:
    try:
        beacon.stop()
    except Exception as exc:
        print(f"Beacon stop warning: {exc}")

await otflex.disconnect()
print("Teardown complete.")